In [ ]:
import cv2
import os
import torch
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO
from pathlib import Path
from collections import defaultdict, deque

# ==========================================
# 1. CONFIGURAZIONE
# ==========================================
NOME_SEQUENZA = "MVI_40905" 
DEVICE = 'cpu'  # Manteniamo CPU per stabilità

# Direzione del traffico nella sequenza: 
# 'AWAY' = le auto si allontanano (vanno verso l'alto, Y diminuisce) -> Tipico di MVI_40905
# 'TOWARDS' = le auto si avvicinano (vanno verso il basso, Y aumenta)
TRAFFIC_DIRECTION = 'AWAY' 

PROJECT_ROOT = Path(os.getcwd()).parent 
TEST_IMAGES_DIR = PROJECT_ROOT / "data_processed" / "test" / "images"
RESULTS_DIR = PROJECT_ROOT / "results" / "pass_on_right"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. MONITOR SORPASSO A DESTRA
# ==========================================
class PassOnRightMonitor:
    def __init__(self, direction='AWAY'):
        self.direction = direction
        # Memorizza chi è davanti a chi: {(id_A, id_B): 'ahead' o 'behind'}
        self.relative_positions = {}
        # Posizione corrente per calcoli: {id: (cx, cy)}
        self.current_pos = {}
        # Timer per mostrare l'alert a video (id -> frames_remaining)
        self.offense_timers = defaultdict(int)

    def update(self, tracks):
        if tracks.boxes is None or tracks.boxes.id is None:
            return

        ids = tracks.boxes.id.cpu().numpy().astype(int)
        boxes = tracks.boxes.xyxy.cpu().numpy()
        
        # 1. Aggiorna posizioni correnti
        self.current_pos = {}
        active_ids = []
        for track_id, box in zip(ids, boxes):
            x1, y1, x2, y2 = box
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            self.current_pos[track_id] = (cx, cy)
            active_ids.append(track_id)
            
            # Decrementa timer visualizzazione infrazioni passate
            if self.offense_timers[track_id] > 0:
                self.offense_timers[track_id] -= 1

        # 2. Logica a coppie
        # Confrontiamo ogni veicolo con ogni altro veicolo presente
        for i in range(len(active_ids)):
            for j in range(i + 1, len(active_ids)):
                id_a = active_ids[i]
                id_b = active_ids[j]
                self._check_pair(id_a, id_b)

    def _check_pair(self, id_a, id_b):
        pos_a = self.current_pos[id_a]
        pos_b = self.current_pos[id_b]
        
        # Determina chi è "avanti" in base alla direzione del traffico
        if self.direction == 'AWAY':
            # Se vanno via (in alto), Y minore significa più avanti
            a_is_ahead = pos_a[1] < pos_b[1]
        else:
            # Se vengono verso di noi (in basso), Y maggiore significa più avanti
            a_is_ahead = pos_a[1] > pos_b[1]

        current_status = 'ahead' if a_is_ahead else 'behind'
        
        # Chiave univoca per la coppia (sempre ordinata per coerenza)
        pair_key = tuple(sorted((id_a, id_b)))

        # Se conoscevamo già questa coppia
        if pair_key in self.relative_positions:
            prev_status = self.relative_positions[pair_key]
            
            # SE LO STATO È CAMBIATO -> SORPASSO AVVENUTO
            if prev_status != current_status:
                # Chi ha fatto il sorpasso?
                passer_id = id_a if current_status == 'ahead' else id_b
                passed_id = id_b if passer_id == id_a else id_a
                
                # Controllo Laterale (Chi è a destra?)
                # Assumiamo sempre che X cresca verso destra
                passer_x = self.current_pos[passer_id][0]
                passed_x = self.current_pos[passed_id][0]
                
                if passer_x > passed_x:
                    # IL PASSER HA SORPASSATO ED È A DESTRA -> INFRAZIONE
                    # Attiva il timer per 50 frames (2 secondi)
                    self.offense_timers[passer_id] = 50
                    print(f"⚠️ Frame detection: Veicolo {passer_id} sorpassa a destra veicolo {passed_id}")

        # Aggiorna lo stato per il prossimo frame
        self.relative_positions[pair_key] = current_status

# ==========================================
# 3. FUNZIONE PRINCIPALE
# ==========================================
def run_pass_right_test():
    print(f"🚀 Avvio Analisi Sorpasso a Destra su: {NOME_SEQUENZA}")

    # Carica Modello
    try:
        custom_model = PROJECT_ROOT / 'runs/detect/yolov8n_vehicle_detection3/weights/best.pt'
        model_path = custom_model if custom_model.exists() else "yolov8n.pt"
        model = YOLO(str(model_path))
    except Exception as e:
        print(f"❌ Errore modello: {e}")
        return

    # Carica Immagini
    seq_path = TEST_IMAGES_DIR / NOME_SEQUENZA
    if not seq_path.exists():
        print(f"❌ Sequenza non trovata: {seq_path}")
        return
    images = sorted(list(seq_path.glob("*.jpg")) + list(seq_path.glob("*.png")))
    
    # Video Writer
    h, w = cv2.imread(str(images[0])).shape[:2]
    out_path = RESULTS_DIR / f"pass_right_{NOME_SEQUENZA}.avi"
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*'MJPG'), 25, (w, h))

    monitor = PassOnRightMonitor(direction=TRAFFIC_DIRECTION)

    print("▶️ Elaborazione in corso...")
    for img_path in tqdm(images):
        frame = cv2.imread(str(img_path))
        
        # Tracking
        results = model.track(frame, persist=True, verbose=False, device=DEVICE, tracker="bytetrack.yaml")[0]
        
        # Aggiorna Logica
        monitor.update(results)
        
        # Disegno
        if results.boxes.id is not None:
            boxes = results.boxes.xyxy.cpu().numpy()
            ids = results.boxes.id.cpu().numpy().astype(int)
            
            for box, track_id in zip(boxes, ids):
                x1, y1, x2, y2 = map(int, box)
                
                # Default: Verde
                color = (0, 255, 0)
                label = f"ID:{track_id}"
                thickness = 2
                
                # Se è colpevole di sorpasso a destra (timer > 0)
                if monitor.offense_timers[track_id] > 0:
                    color = (0, 0, 255) # Rosso
                    label += " RX-PASS!"
                    thickness = 4 # Box più spesso
                
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        writer.write(frame)

    writer.release()
    print(f"\n✅ Video salvato in: {out_path}")

run_pass_right_test()

🚀 Avvio Analisi Sorpasso a Destra su: MVI_40905
▶️ Elaborazione in corso...


  0%|          | 3/1710 [00:00<03:22,  8.45it/s]

⚠️ Frame detection: Veicolo 15 sorpassa a destra veicolo 23
⚠️ Frame detection: Veicolo 22 sorpassa a destra veicolo 4
⚠️ Frame detection: Veicolo 21 sorpassa a destra veicolo 26


  0%|          | 7/1710 [00:00<02:48, 10.13it/s]

⚠️ Frame detection: Veicolo 6 sorpassa a destra veicolo 16


...

⚠️ Frame detection: Veicolo 21 sorpassa a destra veicolo 256
⚠️ Frame detection: Veicolo 12 sorpassa a destra veicolo 381


 18%|█▊        | 313/1710 [00:28<02:09, 10.81it/s]

⚠️ Frame detection: Veicolo 256 sorpassa a destra veicolo 35
⚠️ Frame detection: Veicolo 9 sorpassa a destra veicolo 7
⚠️ Frame detection: Veicolo 10 sorpassa a destra veicolo 26


 19%|█▊        | 317/1710 [00:28<02:09, 10.79it/s]

⚠️ Frame detection: Veicolo 440 sorpassa a destra veicolo 35
⚠️ Frame detection: Veicolo 391 sorpassa a destra veicolo 95
